# 🧠 Transfer Learning & Customização de Modelo com Ollama

Este notebook demonstra o processo de **Transfer Learning / Customização de Modelos de Linguagem (LLMs)** utilizando a biblioteca **Ollama** em Python, baseado nos conceitos de transferência de estilo, persona e conhecimento comportamental apresentados no tutorial:

> *"Criando um chatbot com Llama3: da coleta de dados ao desenvolvimento da interface"* (Paulo Cotta)

---

### 🎯 Objetivos:
1. **Conexão com o Ollama:** Interação com o servidor local através da biblioteca oficial `ollama` em Python.
2. **Ingestão do Dataset Processado:** Carregamento das 25 conversas autênticas geradas em `data/processed/chat_dataset_sample25.jsonl`.
3. **Construção Dinâmica do Modelfile:** Definição da arquitetura de adaptação comportamental:
   - **`FROM`**: Modelo base foundation (ex: `llama3.2`, `llama3:8b` ou `llama3`).
   - **`PARAMETER`**: Ajuste fino de hiperparâmetros de decodificação (`temperature`, `top_p`, `stop tokens`).
   - **`SYSTEM`**: Prompt de sistema definindo a identidade, tom informal e regras do clone digital.
   - **`MESSAGE`**: Injeção estruturada de exemplos conversacionais (*in-context few-shot learning*) extraídos do dataset real.
4. **Criação do Modelo Personalizado:** Criação do modelo `yan-clone` programaticamente via `ollama.create()`.
5. **Avaliação e Testes de Inferência:** Comparação das respostas geradas pelo clone frente a prompts do cotidiano.
6. **Interface de Chat Interativa:** Construção de uma interface de conversação similar ao artigo (suporte a Gradio e chat interativo).

In [ ]:
import sys
from pathlib import Path

# Configurar caminho raiz do projeto
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import ollama
from personal_assistant.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR

print(f"Raiz do projeto: {project_root}")
print(f"Diretório de dados processados: {PROCESSED_DATA_DIR}")
print(f"Diretório de modelos: {MODELS_DIR}")
print(f"Biblioteca Ollama carregada com sucesso!")

## 1. Conexão com o Servidor Ollama e Verificação de Modelos

A biblioteca Python `ollama` comunica-se diretamente com o serviço local do Ollama (`http://localhost:11434`).
Vamos verificar os modelos disponíveis localmente no ambiente.

In [ ]:
available_models = []

try:
    client = ollama.Client()
    response = client.list()
    available_models = [m.model for m in response.models]
    print("✅ Conexão com o Ollama estabelecida com sucesso!")
    print(f"Modelos encontrados ({len(available_models)}):")
    for m in available_models:
        print(f"  - {m}")
except Exception as e:
    print("⚠️ Não foi possível listar os modelos do Ollama automaticamente.")
    print(f"Detalhes: {e}")
    print("\nSe o daemon do Ollama ainda não estiver rodando em segundo plano, inicie-o na máquina.")

## 2. Carregamento dos Dados Processados para o Clone Digital

Carregamos o dataset de 25 conversas diversificadas e limpas gerado em `data/processed/chat_dataset_sample25.jsonl`.

In [ ]:
dataset_path = PROCESSED_DATA_DIR / "chat_dataset_sample25.jsonl"

if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset não encontrado em: {dataset_path}. Execute o pipeline primeiro!")

with open(dataset_path, "r", encoding="utf-8") as f:
    dataset_dialogues = [json.loads(line) for line in f]

print(f"Total de conversas carregadas: {len(dataset_dialogues)}\n")

# Exibir um exemplo completo de diálogo
sample_dialogue = dataset_dialogues[0]
print("--- Exemplo de Diálogo Carregado ---")
for msg in sample_dialogue["messages"]:
    print(f"[{msg['role'].upper()}]:\n{msg['content']}\n")

## 3. Construção do Modelfile para Transfer Learning

No ecossistema Ollama, o **Modelfile** é o artefato central que configura a adaptação do modelo:

1. **`FROM`**: Especifica o modelo base pré-treinado (ex: `llama3.2`, `llama3:8b` ou `llama3`).
2. **`PARAMETER`**: Calibra hiperparâmetros:
   - `temperature 0.7`: Equilíbrio ideal entre espontaneidade conversacional e coerência factual.
   - `top_p 0.9`: Amostragem por núcleo para maior naturalidade.
   - `stop`: Tokens de término para evitar repetições desnecessárias.
3. **`SYSTEM`**: Instrução de persona que define o clone digital de Yan Chagas.
4. **`MESSAGE`**: Exemplos de turnos `user` e `assistant` alimentados a partir das conversas processadas, permitindo que o modelo aprenda o tom de voz e vocabulário autêntico de Yan Chagas.

In [ ]:
def build_modelfile_content(
    base_model: str = "llama3.2",
    clone_name: str = "Yan Chagas",
    temperature: float = 0.7,
    top_p: float = 0.9,
    dialogues: list = None,
    max_dialogues: int = 15,
) -> str:
    """Gera o conteúdo completo de um Modelfile para o Ollama com few-shot in-context learning."""
    system_prompt = (
        f"Você é o clone digital de {clone_name}. "
        f"Responda sempre direto ao ponto, de forma autêntica, descontraída e informal, "
        f"exatamente como nas conversas com seus amigos. "
        f"Use linguagem natural brasileira, gírias autênticas quando couber (ex: 'zorra', 'massa', 'tranquilo', 'tô na correria'), "
        f"evite explicações longas ou corporativas e não pareça um assistente robótico."
    )

    lines = [
        f"FROM {base_model}",
        f"PARAMETER temperature {temperature}",
        f"PARAMETER top_p {top_p}",
        'PARAMETER stop "<|eot_id|>"',
        'PARAMETER stop "<|end_of_text|>"',
        "",
        f'SYSTEM """{system_prompt}"""',
        "",
        "# =======================================================",
        "# Exemplos Conversacionais de Transfer Learning (Few-Shot)",
        "# =======================================================",
    ]

    if dialogues:
        included = 0
        for d in dialogues:
            for msg in d.get("messages", []):
                role = msg.get("role")
                if role in ["user", "assistant"]:
                    # Escapar aspas duplas no corpo da mensagem
                    sanitized = msg["content"].replace('"', '\\"')
                    lines.append(f'MESSAGE {role} "{sanitized}"')
            included += 1
            if included >= max_dialogues:
                break

    return "\n".join(lines)


# Seleção do modelo base (usa llama3.2 se disponível, ou llama3 como padrão)
default_base = "llama3.2" if any("llama3.2" in m for m in available_models) else "llama3"

modelfile_text = build_modelfile_content(
    base_model=default_base,
    clone_name="Yan Chagas",
    temperature=0.7,
    top_p=0.9,
    dialogues=dataset_dialogues,
    max_dialogues=15
)

# Salvar arquivo Modelfile no diretório models/
modelfile_output_path = MODELS_DIR / "Modelfile.yan_clone"
with open(modelfile_output_path, "w", encoding="utf-8") as f:
    f.write(modelfile_text)

print(f"✅ Modelfile gerado com sucesso em: {modelfile_output_path}")
print(f"Total de linhas do Modelfile: {len(modelfile_text.splitlines())}")

### Prévia das Primeiras Linhas do Modelfile Gerado

In [ ]:
print("\n".join(modelfile_text.splitlines()[:30]))

## 4. Criação do Modelo Personalizado via Biblioteca Ollama em Python

Utilizamos `ollama.create()` para registrar e construir o novo modelo customizado `yan-clone` no servidor local.

In [ ]:
custom_model_tag = "yan-clone"

try:
    print(f"Iniciando compilação do modelo '{custom_model_tag}' com a biblioteca Ollama...")
    
    # Criação do modelo com streaming de progresso
    progress_stream = ollama.create(
        model=custom_model_tag,
        modelfile=modelfile_text,
        stream=True
    )
    
    for step in progress_stream:
        status_msg = step.get("status", "")
        if status_msg:
            print(f"Status: {status_msg}")
            
    print(f"\n🎉 Modelo '{custom_model_tag}' registrado e pronto para uso!")
except Exception as err:
    print(f"⚠️ Aviso durante criação via API: {err}")
    print("\nDica: você também pode compilar via terminal com o comando:")
    print(f"ollama create {custom_model_tag} -f {modelfile_output_path}")

## 5. Avaliação e Testes de Inferência

Testamos o clone com perguntas típicas do dia a dia (alimentação, faculdade/trabalho, tecnologia e lazer), validando se as respostas são diretas e preservam o estilo informal de Yan Chagas.

In [ ]:
test_prompts = [
    "Bora almoçar onde hoje?",
    "Conseguiu ver aquele erro no banco de dados?",
    "Vai ter aula de que hoje na faculdade?",
    "Bora jogar um CS mais tarde?",
]

def query_clone(prompt: str, model_name: str = custom_model_tag) -> str:
    """Envia uma mensagem ao modelo customizado e retorna a resposta gerada."""
    try:
        res = ollama.chat(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return res["message"]["content"]
    except Exception as e:
        return f"[Erro ao consultar {model_name}: {e}]"

print("=== Bateria de Testes com o Clone Digital ===\n")
for prompt in test_prompts:
    print(f"💬 Usuário: {prompt}")
    resposta = query_clone(prompt)
    print(f"🤖 {custom_model_tag}:\n{resposta}")
    print("-" * 60)

## 6. Comparação Lado a Lado: Modelo Base vs. Clone Digital

Para observar a eficácia do Transfer Learning / In-Context Adaptation, comparamos a resposta de um modelo genérico (assistente formal) com a resposta do clone digital customizado.

In [ ]:
comparison_prompt = "E aí, vale a pena comprar esse PC gamer por 5 mil?"

print(f"Pergunta Comparativa: \"{comparison_prompt}\"\n")

# Resposta do Clone Digital
resp_clone = query_clone(comparison_prompt, model_name=custom_model_tag)
print(f"--- Resposta do Clone ({custom_model_tag}) ---")
print(resp_clone)
print("\n" + "=" * 60 + "\n")

# Resposta do Modelo Base (se disponível)
resp_base = query_clone(comparison_prompt, model_name=default_base)
print(f"--- Resposta do Modelo Base ({default_base}) ---")
print(resp_base)

## 7. Interface Interativa de Chat (Estilo Gradio / Chatbot)

Como demonstrado no tutorial de referência de Paulo Cotta, a etapa final consiste em prover uma interface amigável para que qualquer usuário possa conversar em tempo real com o clone.

Abaixo disponibilizamos a integração com **Gradio**, com fallback automático para modo interativo em linha de comando/notebook caso a biblioteca Gradio ainda não esteja instalada.

In [ ]:
def chat_with_clone(message: str, history: list) -> str:
    """Função de callback para a interface de chat mantendo histórico."""
    messages = []
    for user_turn, asst_turn in history:
        messages.append({"role": "user", "content": user_turn})
        messages.append({"role": "assistant", "content": asst_turn})
    messages.append({"role": "user", "content": message})
    
    try:
        res = ollama.chat(model=custom_model_tag, messages=messages)
        return res["message"]["content"]
    except Exception as e:
        return f"Erro na comunicação com Ollama: {e}"

# Verificar se Gradio está instalado
try:
    import gradio as gr
    
    interface = gr.ChatInterface(
        fn=chat_with_clone,
        title="🤖 Clone Digital de Yan Chagas",
        description="Converse com o clone digital criado via Transfer Learning com Ollama e LLaMA 3.",
        examples=[
            "Bora almoçar onde hoje?",
            "Conseguiu ver aquele erro no banco?",
            "Bora jogar um game mais tarde?",
            "Onde vai ser a aula hoje?"
        ]
    )
    print("✅ Gradio detectado! Para lançar a interface web, execute: interface.launch()")
    # interface.launch()
except ImportError:
    print("ℹ️ Gradio não está instalado no ambiente virtual.")
    print("Para testar interativamente no terminal, você pode chamar diretamente:")
    print("chat_with_clone('sua mensagem aqui', history=[])")
    print("\nPara instalar o Gradio futuramente: uv add gradio")

## 8. Conclusão e Resumo do Pipeline

Com este pipeline concluído:
1. **Dados:** Transformamos o histórico do WhatsApp em 25 diálogos limpos e representativos em `.jsonl`.
2. **Transfer Learning:** Adaptamos o modelo base da família LLaMA 3 via Modelfile no Ollama, transferindo gírias, brevidade e o estilo de Yan Chagas.
3. **Pronto para Produção:** O modelo `yan-clone` fica salvo localmente no Ollama e pode ser invocado via API Python, CLI (`ollama run yan-clone`) ou por interfaces como Gradio e Streamlit.